# 🔬 tango vs AdamW vs SAM — MNIST Benchmark

**Setup:** Go to `Runtime → Change runtime type → T4 GPU` before running.

This notebook benchmarks the **tango v5 optimizer** against a best-over-grid **AdamW** baseline
and **SAM (Sharpness-Aware Minimization)** on MNIST image classification.

- Same ConvNet architecture for all optimizers
- AdamW is given its best LR from a 5-point grid search (fair comparison)
- SAM uses the same LR grid search with its base AdamW optimizer
- tango uses its fixed explore/exploit schedule — no tuning
- Best-checkpoint restore after training
- Results plotted inline


In [ ]:
# ── Cell 1: Imports & device ──────────────────────────────────────────────────
import copy, json, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU found — running on CPU (will be slower). '
          'Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Config — edit here ────────────────────────────────────────────────
N_EPOCHS   = 10      # epochs per run
N_SEEDS    = 5       # seeds (5 = full benchmark, 2 = quick test)
BATCH_SIZE = 128
DATA_DIR   = './data'
SUBSET     = 0       # 0 = full 60k; set e.g. 5000 for a faster stress-test

ADAM_LR_GRID = [1e-2, 3e-3, 1e-3, 3e-4, 1e-4]

# ── tango v5 hyperparams (DO NOT CHANGE — identical to raw-fn benchmark) ──────
LR_MAX           = 0.005
LR_MIN           = 0.000001
T_CYCLE          = 800
EXPLORE_FRAC     = 0.8
LR_EXPLOIT_START = 0.005
LR_EXPLOIT_END   = 0.0005
EXPLORE_DECAY_START = 0.6
EXPLORE_FLOOR       = 0.03
TANG_BETA        = 0.80
TANG_EPS_BASE    = 4e-4
TANG_INTERVAL    = 100
TANG_LOSS_GATE   = 0
SHARP_UPDATE_INT = 200
FD_EPS           = 1e-4
NOISE_SCALE      = 2e-4
NOISE_START      = 500
print('Config loaded.')

In [ ]:
# ── Cell 3: Model & data ──────────────────────────────────────────────────────
class SmallConvNet(nn.Module):
    """Two conv layers + two FC layers. ~93k params. No BatchNorm."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(32 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


def get_loaders(seed):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_full = datasets.MNIST(DATA_DIR, train=True,  download=True, transform=transform)
    test_ds    = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)
    if SUBSET > 0:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(train_full), SUBSET, replace=False).tolist()
        train_ds = Subset(train_full, idx)
    else:
        train_ds = train_full
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=(DEVICE.type=='cuda'), generator=g)
    test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False,
                              num_workers=2, pin_memory=(DEVICE.type=='cuda'))
    return train_loader, test_loader

print('Model & data helpers ready.')

In [ ]:
# ── Cell 4: tango v5 core (schedule, curvature, tangent momentum) ─────────────
def cyclic_lr(step, T, lr_max, lr_min):
    t = step % T
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * t / T))

def cosine_decay_lr(step, total_steps, lr_start, lr_end):
    progress = min(step / max(total_steps, 1), 1.0)
    return lr_end + 0.5 * (lr_start - lr_end) * (1 + np.cos(np.pi * progress))

def get_exploration_factor(step, total_steps, decay_start_frac, floor):
    decay_start = int(total_steps * decay_start_frac)
    if step < decay_start:
        return 1.0
    dp = (step - decay_start) / max(total_steps - decay_start, 1)
    return max(floor, 1.0 - dp)

def get_flat_grad(model):
    grads = []
    for p in model.parameters():
        grads.append(p.grad.detach().cpu().view(-1).numpy() if p.grad is not None
                     else np.zeros(p.numel(), dtype=np.float32))
    return np.concatenate(grads).astype(np.float32)

def get_flat_params(model):
    return np.concatenate(
        [p.detach().cpu().view(-1).numpy() for p in model.parameters()]
    ).astype(np.float32)

def set_flat_params(model, flat):
    offset = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(torch.tensor(flat[offset:offset+n], dtype=p.dtype).view(p.shape))
        offset += n

def fd_curvature_nn(model, loss_fn, bx, by, g_np, eps=FD_EPS):
    norm_g = np.linalg.norm(g_np)
    if norm_g < 1e-12:
        return 0.0
    g_hat   = g_np / norm_g
    params0 = get_flat_params(model)
    with torch.no_grad():
        f0    = loss_fn(model(bx), by).item()
    set_flat_params(model, params0 + g_hat * eps)
    with torch.no_grad():
        f_eps = loss_fn(model(bx), by).item()
    set_flat_params(model, params0)
    return float((f_eps - f0 - eps * norm_g) / (0.5 * eps**2 + 1e-30))

class TangentMomentum:
    def __init__(self, dim, beta=TANG_BETA):
        self.beta = beta
        self.v    = np.zeros(dim, dtype=np.float32)
    def step(self, model, g_np, epsilon):
        g_hat = g_np / (np.linalg.norm(g_np) + 1e-8)
        noise = np.random.randn(len(g_np)).astype(np.float32)
        noise -= np.dot(noise, g_hat) * g_hat
        noise /= np.linalg.norm(noise) + 1e-8
        self.v = self.beta * self.v + (1 - self.beta) * noise
        direction = self.v / (np.linalg.norm(self.v) + 1e-8)
        set_flat_params(model, get_flat_params(model) + direction * epsilon)

def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb  = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            total_loss += loss_fn(logits, yb).item() * yb.size(0)
            correct    += (logits.argmax(1) == yb).sum().item()
            total      += yb.size(0)
    model.train()
    return total_loss / total, correct / total

print('tango v5 core ready.')

In [ ]:
# ── Cell 5: run_tango & run_adam ──────────────────────────────────────────────
def run_tango(seed, train_loader, test_loader):
    torch.manual_seed(seed)
    model    = SmallConvNet().to(DEVICE)
    loss_fn  = nn.CrossEntropyLoss()
    opt      = torch.optim.AdamW(model.parameters(), lr=LR_MAX,
                                  betas=(0.9, 0.999), weight_decay=0.0)
    n_params = sum(p.numel() for p in model.parameters())
    tang_mom = TangentMomentum(dim=n_params)

    steps_per_epoch = len(train_loader)
    total_steps     = N_EPOCHS * steps_per_epoch
    explore_end     = int(total_steps * EXPLORE_FRAC)

    best_val_loss = float('inf')
    best_state    = copy.deepcopy(model.state_dict())
    sharpness     = 0.0
    tang_exec = tang_block = step = 0
    history = []

    for epoch in range(N_EPOCHS):
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            # LR schedule
            if step < explore_end:
                lr = cyclic_lr(step, T_CYCLE, LR_MAX, LR_MIN)
            else:
                lr = cosine_decay_lr(step - explore_end, total_steps - explore_end,
                                     LR_EXPLOIT_START, LR_EXPLOIT_END)
            for pg in opt.param_groups: pg['lr'] = lr
            ef = get_exploration_factor(step, total_steps, EXPLORE_DECAY_START, EXPLORE_FLOOR)
            # gradient step
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            g_np = get_flat_grad(model)
            opt.step()
            loss_val = loss.item()
            # noise
            if step >= NOISE_START:
                sigma = NOISE_SCALE * np.sqrt(max(loss_val, 1e-8)) * ef
                with torch.no_grad():
                    for p in model.parameters(): p.data.add_(torch.randn_like(p) * sigma)
            # sharpness
            if step % SHARP_UPDATE_INT == 0 and step > 0:
                sharpness = fd_curvature_nn(model, loss_fn, xb, yb, g_np)
            # tangent step
            if step % TANG_INTERVAL == 0 and step > 500:
                loss_gate  = loss_val > TANG_LOSS_GATE * max(best_val_loss, 1e-10)
                sharp_gate = sharpness < 0.8 * (2.0 / max(lr, 1e-8))
                if loss_gate and sharp_gate:
                    sharp_safe = max(abs(sharpness), 1.0)
                    eps_tang   = float(np.clip(TANG_EPS_BASE * (50.0/sharp_safe)**0.5, 1e-5, 5e-3)) * ef
                    tang_mom.step(model, g_np, eps_tang)
                    tang_exec += 1
                else:
                    tang_block += 1
            step += 1

        val_loss, val_acc = evaluate(model, test_loader, loss_fn)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = copy.deepcopy(model.state_dict())
        history.append({'epoch': epoch+1, 'val_loss': val_loss, 'val_acc': val_acc,
                        'lr': lr, 'ef': ef})
        print(f'  [tango seed={seed}] epoch {epoch+1:2d}/{N_EPOCHS}  '
              f'val_loss={val_loss:.4f}  val_acc={val_acc*100:.2f}%  '
              f'lr={lr:.2e}  ef={ef:.2f}')

    model.load_state_dict(best_state)
    final_val_loss, final_val_acc = evaluate(model, test_loader, loss_fn)
    return {'optimizer': 'tango_v5', 'seed': seed,
            'final_val_loss': float(final_val_loss),
            'final_val_acc':  float(final_val_acc),
            'best_val_loss':  float(best_val_loss),
            'tang_exec': tang_exec, 'tang_block': tang_block,
            'history': history}


def run_adam_single(seed, lr, train_loader, test_loader):
    torch.manual_seed(seed)
    model   = SmallConvNet().to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    opt     = torch.optim.AdamW(model.parameters(), lr=lr,
                                 betas=(0.9, 0.999), weight_decay=0.0)
    history = []
    for epoch in range(N_EPOCHS):
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
        val_loss, val_acc = evaluate(model, test_loader, loss_fn)
        history.append({'epoch': epoch+1, 'val_loss': val_loss, 'val_acc': val_acc})
    return {'optimizer': f'adam_lr{lr:.0e}', 'lr': lr, 'seed': seed,
            'final_val_loss': float(val_loss),
            'final_val_acc':  float(val_acc),
            'history': history}

def run_adam_best(seed, train_loader, test_loader):
    best = None
    for lr in ADAM_LR_GRID:
        r = run_adam_single(seed, lr, train_loader, test_loader)
        print(f'  [Adam  seed={seed} lr={lr:.0e}]  '
              f'val_loss={r["final_val_loss"]:.4f}  val_acc={r["final_val_acc"]*100:.2f}%')
        if best is None or r['final_val_loss'] < best['final_val_loss']:
            best = r
    return best

print('Runner functions ready.')

In [ ]:
# ── Cell 5b: SAM implementation & run_sam ────────────────────────────────────
class SAM(torch.optim.Optimizer):
    """Sharpness-Aware Minimization (Foret et al., 2021).

    Wraps any base optimizer. Each step does:
      1. Forward + backward → compute gradient.
      2. Perturb weights by ε * g / ||g||  (ascent step).
      3. Forward + backward on perturbed weights.
      4. Restore original weights, apply base optimizer with new gradient.
    """
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        """Gradient ascent step: perturb weights toward sharp region."""
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group['rho'] / (grad_norm + 1e-12)
            for p in group['params']:
                if p.grad is None:
                    continue
                e_w = p.grad * scale
                p.add_(e_w)            # perturb
                self.state[p]['e_w'] = e_w
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        """Restore weights and apply base optimizer update."""
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                p.sub_(self.state[p]['e_w'])  # un-perturb
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    def _grad_norm(self):
        shared_device = self.param_groups[0]['params'][0].device
        norm = torch.norm(
            torch.stack([
                p.grad.norm(p=2).to(shared_device)
                for group in self.param_groups
                for p in group['params']
                if p.grad is not None
            ]),
            p=2
        )
        return norm


def run_sam_single(seed, lr, train_loader, test_loader, rho=0.05):
    """Run one SAM trial with a fixed LR (AdamW base optimizer)."""
    torch.manual_seed(seed)
    model   = SmallConvNet().to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = SAM(
        model.parameters(),
        torch.optim.AdamW,
        rho=rho,
        lr=lr,
        betas=(0.9, 0.999),
        weight_decay=0.0
    )
    history = []
    for epoch in range(N_EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            # ── First forward/backward (for ascent step) ──────────────────────
            loss_fn(model(xb), yb).backward()
            optimizer.first_step(zero_grad=True)
            # ── Second forward/backward (on perturbed weights) ────────────────
            loss_fn(model(xb), yb).backward()
            optimizer.second_step(zero_grad=True)
        val_loss, val_acc = evaluate(model, test_loader, loss_fn)
        history.append({'epoch': epoch + 1, 'val_loss': val_loss, 'val_acc': val_acc})
    return {
        'optimizer': f'sam_lr{lr:.0e}',
        'lr': lr, 'rho': rho, 'seed': seed,
        'final_val_loss': float(val_loss),
        'final_val_acc':  float(val_acc),
        'history': history
    }


def run_sam_best(seed, train_loader, test_loader, rho=0.05):
    """Grid-search the same LR grid as AdamW, pick the best SAM run."""
    best = None
    for lr in ADAM_LR_GRID:
        r = run_sam_single(seed, lr, train_loader, test_loader, rho=rho)
        print(f'  [SAM   seed={seed} lr={lr:.0e}]  '
              f'val_loss={r["final_val_loss"]:.4f}  val_acc={r["final_val_acc"]*100:.2f}%')
        if best is None or r['final_val_loss'] < best['final_val_loss']:
            best = r
    return best


print('SAM implementation & runner ready.')


In [ ]:
# ── Cell 6: RUN ───────────────────────────────────────────────────────────────
print('=' * 68)
print('  tango BENCHMARK SUITE — MNIST EDITION')
print(f'  {N_EPOCHS} epochs | {N_SEEDS} seeds | {len(ADAM_LR_GRID)} LRs | device={DEVICE}')
print('=' * 68)

tango_all, adam_all, sam_all = [], [], []
u_losses, a_losses, s_losses = [], [], []
u_accs,   a_accs,   s_accs   = [], [], []
grand_start = time.time()

for seed in range(N_SEEDS):
    print(f'\n{"─"*68}\n  SEED {seed}\n{"─"*68}')
    t0 = time.time()
    train_loader, test_loader = get_loaders(seed)

    ur = run_tango(seed, train_loader, test_loader)
    tango_all.append(ur); u_losses.append(ur['final_val_loss']); u_accs.append(ur['final_val_acc'])

    ar = run_adam_best(seed, train_loader, test_loader)
    adam_all.append(ar); a_losses.append(ar['final_val_loss']); a_accs.append(ar['final_val_acc'])

    sr = run_sam_best(seed, train_loader, test_loader)
    sam_all.append(sr); s_losses.append(sr['final_val_loss']); s_accs.append(sr['final_val_acc'])

    def _tag(u, a, s):
        best = min(u, a, s)
        return ('U' if best == u else ('A' if best == a else 'S'))

    tag = _tag(ur['final_val_loss'], ar['final_val_loss'], sr['final_val_loss'])
    print(f'\n  seed={seed}  '
          f'tango loss={ur["final_val_loss"]:.4f} acc={ur["final_val_acc"]*100:.2f}%  ||  '
          f'Adam  loss={ar["final_val_loss"]:.4f} acc={ar["final_val_acc"]*100:.2f}%  ||  '
          f'SAM   loss={sr["final_val_loss"]:.4f} acc={sr["final_val_acc"]*100:.2f}%  '
          f'[{tag}]  {time.time()-t0:.1f}s')

print(f'\nTotal wall time: {time.time()-grand_start:.1f}s')


In [ ]:
# ── Cell 7: Summary stats ─────────────────────────────────────────────────────
import math
u_arr = np.array(u_losses); a_arr = np.array(a_losses); s_arr = np.array(s_losses)
u_acc = np.array(u_accs);   a_acc = np.array(a_accs);   s_acc = np.array(s_accs)

wins_loss_u = int(np.sum((u_arr < a_arr) & (u_arr < s_arr)))
wins_loss_a = int(np.sum((a_arr < u_arr) & (a_arr < s_arr)))
wins_loss_s = int(np.sum((s_arr < u_arr) & (s_arr < a_arr)))
wins_acc_u  = int(np.sum((u_acc > a_acc) & (u_acc > s_acc)))
wins_acc_a  = int(np.sum((a_acc > u_acc) & (a_acc > s_acc)))
wins_acc_s  = int(np.sum((s_acc > u_acc) & (s_acc > a_acc)))

total_exec  = sum(r['tang_exec']  for r in tango_all)
total_block = sum(r['tang_block'] for r in tango_all)

print('=' * 68)
print('  MNIST FINAL SUMMARY')
print('=' * 68)
print(f'  tango wins (lower val_loss)  : {wins_loss_u}/{N_SEEDS}')
print(f'  Adam  wins (lower val_loss)  : {wins_loss_a}/{N_SEEDS}')
print(f'  SAM   wins (lower val_loss)  : {wins_loss_s}/{N_SEEDS}')
print()
print(f'  tango wins (higher val_acc)  : {wins_acc_u}/{N_SEEDS}')
print(f'  Adam  wins (higher val_acc)  : {wins_acc_a}/{N_SEEDS}')
print(f'  SAM   wins (higher val_acc)  : {wins_acc_s}/{N_SEEDS}')
print()
print(f'  tango  loss  mean={np.mean(u_arr):.4f}  std={np.std(u_arr):.4f}  '
      f'min={np.min(u_arr):.4f}  max={np.max(u_arr):.4f}')
print(f'  Adam   loss  mean={np.mean(a_arr):.4f}  std={np.std(a_arr):.4f}  '
      f'min={np.min(a_arr):.4f}  max={np.max(a_arr):.4f}')
print(f'  SAM    loss  mean={np.mean(s_arr):.4f}  std={np.std(s_arr):.4f}  '
      f'min={np.min(s_arr):.4f}  max={np.max(s_arr):.4f}')
print()
print(f'  tango  acc   mean={np.mean(u_acc)*100:.2f}%  std={np.std(u_acc)*100:.2f}%')
print(f'  Adam   acc   mean={np.mean(a_acc)*100:.2f}%  std={np.std(a_acc)*100:.2f}%')
print(f'  SAM    acc   mean={np.mean(s_acc)*100:.2f}%  std={np.std(s_acc)*100:.2f}%')
if total_exec + total_block > 0:
    print(f'  Tangent fire rate: {total_exec/(total_exec+total_block)*100:.1f}%  '
          f'({total_exec} exec / {total_block} blocked)')


In [ ]:
# ── Cell 8: Plots ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0f0f0f')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

tango_COL = '#00e5ff'
ADAM_COL  = '#ff6b6b'
SAM_COL   = '#a8ff3e'
BG        = '#1a1a2e'
GRID_COL  = '#2a2a3e'

def style_ax(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, color='white', fontsize=11, pad=8)
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color(GRID_COL)
    ax.yaxis.grid(True, color=GRID_COL, linewidth=0.5)
    ax.set_axisbelow(True)

# ── 1. Val loss per epoch (mean ± std across seeds) ──────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
style_ax(ax1, 'Validation Loss — mean ± std across seeds')
for results, col, label in [
    (tango_all, tango_COL, 'tango v5'),
    (adam_all,  ADAM_COL,  'AdamW (best LR)'),
    (sam_all,   SAM_COL,   'SAM (best LR)'),
]:
    max_ep = max(len(r['history']) for r in results)
    matrix = np.full((len(results), max_ep), np.nan)
    for i, r in enumerate(results):
        for h in r['history']:
            matrix[i, h['epoch']-1] = h['val_loss']
    mean = np.nanmean(matrix, axis=0)
    std  = np.nanstd(matrix, axis=0)
    epochs = np.arange(1, max_ep+1)
    ax1.plot(epochs, mean, color=col, linewidth=2, label=label)
    ax1.fill_between(epochs, mean-std, mean+std, color=col, alpha=0.15)
ax1.set_xlabel('Epoch', color='#aaaaaa')
ax1.set_ylabel('Val Loss', color='#aaaaaa')
ax1.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor=GRID_COL)

# ── 2. Val accuracy per epoch ─────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
style_ax(ax2, 'Validation Accuracy — mean ± std across seeds')
for results, col, label in [
    (tango_all, tango_COL, 'tango v5'),
    (adam_all,  ADAM_COL,  'AdamW (best LR)'),
    (sam_all,   SAM_COL,   'SAM (best LR)'),
]:
    max_ep = max(len(r['history']) for r in results)
    matrix = np.full((len(results), max_ep), np.nan)
    for i, r in enumerate(results):
        for h in r['history']:
            matrix[i, h['epoch']-1] = h['val_acc'] * 100
    mean = np.nanmean(matrix, axis=0)
    std  = np.nanstd(matrix, axis=0)
    epochs = np.arange(1, max_ep+1)
    ax2.plot(epochs, mean, color=col, linewidth=2, label=label)
    ax2.fill_between(epochs, mean-std, mean+std, color=col, alpha=0.15)
ax2.set_xlabel('Epoch', color='#aaaaaa')
ax2.set_ylabel('Accuracy (%)', color='#aaaaaa')
ax2.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor=GRID_COL)

# ── 3. Final val loss per seed (bar chart) ────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
style_ax(ax3, 'Final Val Loss per Seed')
seeds = np.arange(N_SEEDS)
w = 0.25
ax3.bar(seeds - w, u_losses, w, label='tango v5', color=tango_COL, alpha=0.85)
ax3.bar(seeds,     a_losses, w, label='AdamW',    color=ADAM_COL,  alpha=0.85)
ax3.bar(seeds + w, s_losses, w, label='SAM',      color=SAM_COL,   alpha=0.85)
ax3.set_xticks(seeds)
ax3.set_xticklabels([f'seed {s}' for s in seeds], color='#aaaaaa')
ax3.set_ylabel('Val Loss', color='#aaaaaa')
ax3.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor=GRID_COL)

# ── 4. Final val accuracy per seed ───────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
style_ax(ax4, 'Final Val Accuracy per Seed (%)')
ax4.bar(seeds - w, np.array(u_accs)*100, w, label='tango v5', color=tango_COL, alpha=0.85)
ax4.bar(seeds,     np.array(a_accs)*100, w, label='AdamW',    color=ADAM_COL,  alpha=0.85)
ax4.bar(seeds + w, np.array(s_accs)*100, w, label='SAM',      color=SAM_COL,   alpha=0.85)
ax4.set_xticks(seeds)
ax4.set_xticklabels([f'seed {s}' for s in seeds], color='#aaaaaa')
ax4.set_ylabel('Accuracy (%)', color='#aaaaaa')
bottom_val = max(min(min(u_accs), min(a_accs), min(s_accs))*100 - 1.0, 90.0)
ax4.set_ylim(bottom=bottom_val)
ax4.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor=GRID_COL)

fig.suptitle('tango v5 vs AdamW vs SAM — MNIST Benchmark', color='white',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('mnist_benchmark_results.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Plot saved → mnist_benchmark_results.png')


In [ ]:
# ── Cell 9: Save JSON results ─────────────────────────────────────────────────
meta = {'epochs': N_EPOCHS, 'n_seeds': N_SEEDS, 'batch_size': BATCH_SIZE,
        'adam_lr_grid': ADAM_LR_GRID, 'device': str(DEVICE)}
with open('tango_mnist_results.json', 'w') as f:
    json.dump({'suite': 'tango_mnist_v1', 'meta': meta, 'results': tango_all}, f, indent=2)
with open('adam_mnist_results.json', 'w') as f:
    json.dump({'suite': 'adam_mnist_baseline', 'meta': meta, 'results': adam_all}, f, indent=2)
with open('sam_mnist_results.json', 'w') as f:
    json.dump({'suite': 'sam_mnist_baseline', 'meta': meta, 'results': sam_all}, f, indent=2)
print('Saved → tango_mnist_results.json')
print('Saved → adam_mnist_results.json')
print('Saved → sam_mnist_results.json')

# Optional: download results to your local machine
try:
    from google.colab import files
    files.download('tango_mnist_results.json')
    files.download('adam_mnist_results.json')
    files.download('sam_mnist_results.json')
    files.download('mnist_benchmark_results.png')
except ImportError:
    print('(Not in Colab — skipping download)')
